# Camada Bronze: Projeto CineData Analytics

**Objetivo:** ingerir os dados brutos para o Data Lakehouse, em **formato Delta** e com **modo Append**, sem alterar estrutura nem conteúdo.

**Regras de negócio / decisões desta camada**

| Decisão | Justificativa |
|---|---|
| Todas as colunas lidas como **STRING** inferSchema=False | Como o BD não veio limpo, prefere-se utilizar o tipo string, para não perder dados. |
| multiLine=True, quote='"', escape='"' | Sinopses possuem quebras de linha e aspas internas escapadas. |
| mode("append") | É o que foi solicitado. |
| ingestion_datetime = current_timestamp() | Timestamp do momento da inserção na Bronze (rastreabilidade/linhagem). |

### Pré-requisito feitos
1. No Databricks, criando o **Volume**.
2. Feito o upload dos arquivos do BD.
3. Aplicação dos widgets.

In [0]:
import os
import requests
from datetime import date, datetime, timedelta
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Parâmetros (widgets) do notebook
dbutils.widgets.text("catalogo", "", "") #workspace
dbutils.widgets.text("volume_path", "") #/Volumes/workspace/default/entrega-atv-02/Base_de_Dados
dbutils.widgets.text("data_inicio", "", "") #09-30-2026
dbutils.widgets.text("data_fim", "", "") #10-07-2026

CATALOGO = dbutils.widgets.get("catalogo").strip()
VOLUME_PATH = dbutils.widgets.get("volume_path").strip().rstrip("/")

# Criar o banco de dados (database/schema) da camada Bronze
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
print("Database 'bronze' pronto. Volume de entrada:", VOLUME_PATH)


Database 'bronze' pronto. Volume de entrada: /Volumes/workspace/default/entrega-atv-02/Base_de_Dados


## 1. Ingestão dos CSVs para tabelas Bronze

Mapeamento 

In [0]:
# tabela_bronze -> nomes candidatos do arquivo no Volume
ARQUIVOS = {
    "tb_movies_info":       ["movies_info_IMDB_TMDB.csv", "movies_info_TMDB_IMDB.csv"],
    "tb_movies_financials": ["movies_financials_IMDB_TMDB.csv"],
    "tb_movies_metrics":    ["movies_metics_IMDB_TMDB.csv", "movies_metrics_IMDB_TMDB.csv"],
    "tb_credits_and_tags":  ["credits_and_tags_IMDB_TMDB.csv"],
    "tb_movies_reviews":    ["movies_reviews.csv"],
}


def localizar_arquivo(candidatos):
    # Retorna o caminho do primeiro arquivo candidato que existir no Volume.
    for nome in candidatos:
        caminho = f"{VOLUME_PATH}/{nome}"
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(f"Nenhum dos arquivos {candidatos} foi encontrado em {VOLUME_PATH}")


def ler_csv_bruto(caminho):
    # Leitura: tudo STRING, sem inferência de schema e sem qualquer limpeza.
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", False) # nada de tipos: preserva o conteúdo original
        .option("multiLine", True) # sinopses com quebras de linha
        .option("quote", '"')
        .option("escape", '"') # aspas internas escapadas como ""
        .option("mode", "PERMISSIVE") # linhas malformadas não derrubam a carga
        .load(caminho)
    )


def gravar_bronze(df, tabela):
    # Acrescenta ingestion_datetime e grava em Delta (Append).
    df = df.withColumn("ingestion_datetime", F.current_timestamp())
    (
        df.write.format("delta")
        .mode("append")
        .saveAsTable(f"bronze.{tabela}")
    )


for tabela, candidatos in ARQUIVOS.items():
    caminho = localizar_arquivo(candidatos)
    df_raw = ler_csv_bruto(caminho)
    gravar_bronze(df_raw, tabela)
    print(f"{caminho.split('/')} -> bronze.{tabela}")

['', 'Volumes', 'workspace', 'default', 'entrega-atv-02', 'Base_de_Dados', 'movies_info_TMDB_IMDB.csv'] -> bronze.tb_movies_info
['', 'Volumes', 'workspace', 'default', 'entrega-atv-02', 'Base_de_Dados', 'movies_financials_IMDB_TMDB.csv'] -> bronze.tb_movies_financials
['', 'Volumes', 'workspace', 'default', 'entrega-atv-02', 'Base_de_Dados', 'movies_metrics_IMDB_TMDB.csv'] -> bronze.tb_movies_metrics
['', 'Volumes', 'workspace', 'default', 'entrega-atv-02', 'Base_de_Dados', 'credits_and_tags_IMDB_TMDB.csv'] -> bronze.tb_credits_and_tags
['', 'Volumes', 'workspace', 'default', 'entrega-atv-02', 'Base_de_Dados', 'movies_reviews.csv'] -> bronze.tb_movies_reviews


## 2. Ingestão via API para ver a Cotação do Dólar

A diretoria financeira quer acompanhar orçamento/receita também em BRL. A API devolve a cotação de compra do dólar por período.

* Formato de data exigido pela API: MM-DD-AAAA.
* Datas controladas por widgets data_inicio e data_fim. Se vazios: janela de **7 dias corridos** até hoje (a API não retorna cotação em finais de semana/feriados; uma janela maior que 1 dia garante ao menos um dia útil).
* A API devolve **vários boletins por dia** (intradiários + fechamento). A Bronze guarda tudo como veio; a Silver escolhe o último boletim do dia.

In [0]:
FORMATO_API = "%m-%d-%Y" #formato solicitado


def _data_widget(nome, padrao):
    # Lê o widget (MM-DD-AAAA); se vazio usa o padrão. Valida o formato.
    valor = dbutils.widgets.get(nome).strip()
    if not valor:
        return padrao.strftime(FORMATO_API)
    datetime.strptime(valor, FORMATO_API)  # levanta ValueError se o formato estiver errado
    return valor


hoje = date.today()
data_inicio_formatada = _data_widget("data_inicio", hoje - timedelta(days=7))
data_fim_formatada    = _data_widget("data_fim", hoje)

URL_PTAX = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)
print("Período consultado:", data_inicio_formatada, "a", data_fim_formatada)


def consultar_ptax(url):
    # GET com retry/backoff para erros transitórios.
    sessao = requests.Session()
    sessao.mount("https://", HTTPAdapter(max_retries=Retry(
        total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])))
    resp = sessao.get(url, timeout=60)
    resp.raise_for_status()
    return resp.json().get("value", [])


registros = consultar_ptax(URL_PTAX)
if not registros:
    # Falha explícita, sem cotação a Silver não consegue calcular os valores em BRL.
    raise ValueError("API do Banco Central não retornou cotações para o período.")

# Mantém os valores como STRING, exatamente como a API devolveu.Idea do Bronze em ter dado cru
schema_cotacao = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra",   StringType(), True),
])
linhas = [
    (r.get("dataHoraCotacao"), None if r.get("cotacaoCompra") is None else str(r["cotacaoCompra"]))
    for r in registros
]
df_cotacao = spark.createDataFrame(linhas, schema=schema_cotacao)
gravar_bronze(df_cotacao, "tb_cotacao_dolar")
print(f"API: bronze.tb_cotacao_dolar {len(linhas)} cotações")

Período consultado: 09-30-2026 a 10-07-2026
API: bronze.tb_cotacao_dolar 6 cotações


## 3. Conferência da carga Bronze

In [0]:
resumo = []
for tabela in list(ARQUIVOS) + ["tb_cotacao_dolar"]:
    t = spark.table(f"bronze.{tabela}")
    resumo.append((f"bronze.{tabela}", t.count(), len(t.columns)))

display(spark.createDataFrame(resumo, ["tabela", "qtd_linhas_acumuladas", "qtd_colunas"]))

# Amostra de uma tabela e do schema, todas as colunas de negócio são STRING, e o ingestion_datetime é TIMESTAMP
display(spark.table("bronze.tb_movies_info").limit(5))
spark.table("bronze.tb_movies_info").printSchema()
display(spark.table("bronze.tb_cotacao_dolar").orderBy("dataHoraCotacao"))

tabela,qtd_linhas_acumuladas,qtd_colunas
bronze.tb_movies_info,852768,11
bronze.tb_movies_financials,849320,4
bronze.tb_movies_metrics,824576,7
bronze.tb_credits_and_tags,844864,10
bronze.tb_movies_reviews,259296,5
bronze.tb_cotacao_dolar,70,3


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T21:05:58.066Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T21:05:58.066Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T21:05:58.066Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T21:05:58.066Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T21:05:58.066Z


root
 |-- id: string (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-25 13:10:17.447657,5.1985,2026-10-07T19:45:49.907Z
2026-09-25 13:10:17.447657,5.1985,2026-10-07T19:45:34.044Z
2026-09-28 13:03:11.35885,5.2126,2026-10-07T19:45:49.907Z
2026-09-29 13:06:06.446305,5.2198,2026-10-07T19:45:49.907Z
2026-09-30 13:11:44.783262,5.1803,2026-10-07T21:05:08.531Z
2026-09-30 13:11:44.783262,5.1803,2026-10-07T19:45:49.907Z
2026-09-30 13:11:44.783262,5.1803,2026-10-07T21:17:59.559Z
2026-09-30 13:11:44.783262,5.1803,2026-10-07T20:08:52.662Z
2026-09-30 13:11:44.783262,5.1803,2026-10-07T20:40:43.833Z
2026-09-30 13:11:44.783262,5.1803,2026-10-07T20:12:23.455Z
